## Deep Dyna-Q Tutorial with CartPole

Adapted from [Deep Dyna-Q: Integrating Planning for Task-Completion Dialogue Policy Learning](https://arxiv.org/abs/1801.06176) (Peng et al., 2018, Microsoft Research). The agent learns a world model from real transitions and uses it to generate simulated experience, supplementing real interactions — both contribute to the Q-network's gradients.

The original paper was coupled to a dialogue NLP task. This implementation generalizes the core algorithm to work with any Gymnasium environment.

![Alt text](best_run.png)

In [ ]:
import os
from collections.abc import Callable
from typing import Any

import gymnasium as gym
import numpy as np
import structlog
import torch
import torch.nn as nn
from cart_pole_logger_dyna import CartPoleLoggerDyna
from cart_pole_wm_interpreter import CartPoleInterpreter

from samsara_rl.control.dyna.deep_dyna_q import (
    DeepDynaQ,
)
from samsara_rl.functions.neural_networks.fully_connected import FullyConnected
from samsara_rl.mdp.cart_pole.scaled_cart_pole import ScaledCartPole

log = structlog.get_logger()

%load_ext tensorboard

In [ ]:
cart_pole_wm_interpreter = CartPoleInterpreter()

In [ ]:
log = structlog.get_logger()

MAX_EPISODES = 2000
BASE_DIR = "logs/deep_dyna_q"

ALPHA = 0.0005
GAMMA = 0.99
MODEL_PLAN_STEPS = [0, 2, 5]
WARMUPS = [300, 500, 800]

## Q-Network Architecture

Input: 4 values (position, velocity, angle, angular velocity). Output: 2 Q-values (push left, push right).

Two hidden layers (32 → 16) capture the nonlinear relationship between angle and angular velocity. A single hidden layer can work but requires more careful tuning — the angle/angular velocity dynamics involve sine/cosine curves that benefit from the extra representational capacity.

In [ ]:
class CartPoleQNetwork(nn.Module):
    def __init__(
        self,
        input_dim: int,
        hidden_dim_1: int,
        hidden_dim_2: int,
        output_dim: int,
        preprocess: Callable[..., torch.Tensor] | None = None,
    ) -> None:
        super().__init__()
        self.fc1 = nn.Linear(input_dim, hidden_dim_1, dtype=torch.float64)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim_1, hidden_dim_2, dtype=torch.float64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(hidden_dim_2, output_dim, dtype=torch.float64)
        self.preprocess = preprocess

    def forward(self, x: Any) -> torch.Tensor:
        out: torch.Tensor = self.preprocess(x) if self.preprocess else x
        if isinstance(out, np.ndarray):
            out = torch.from_numpy(out).to(torch.float64)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)
        out = self.relu2(out)
        return self.fc3(out)  # type: ignore[no-any-return]

In [ ]:
def run_experiment(env, model_plan_steps: int, warmup: int) -> None:
    """Train a Deep Dyna-Q agent with fixed hyperparameters, sweeping plan steps and warmup.

    Args:
        env: Gymnasium CartPole environment.
        model_plan_steps: Number of Q-network training steps on simulated data per real transition.
        warmup: Number of episodes before planning begins.
    """
    model_name = "deep_dyna_q"
    run_name = f"{model_name}_plan_steps={model_plan_steps}_warmup={warmup}"
    run_dir = os.path.join(BASE_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    log.info("starting_run", run=run_name)
    cart_pole_wm = FullyConnected(6, 32, 6)
    q = CartPoleQNetwork(4, 32, 16, 2)

    agent = DeepDynaQ(
        mdp=env,
        world_model=cart_pole_wm,
        world_model_interpreter=cart_pole_wm_interpreter,
        model_plan_steps=model_plan_steps,
        count_planning_updates=True,
        model_train_steps=10,
        warmup=warmup,
        gamma=GAMMA,
        q=q,
        alpha=ALPHA,
        state_loss=torch.nn.MSELoss(),
        target_update_freq=500,
        epsilon_episodes=800,
        epsilon_min=0.01,
        lr_step_size=100,
        lr_gamma=0.3,
        log_dir=run_dir,
        experiment_name=run_name,
        batch_size=128,
        buffer_size=10000,
    )
    agent.register(CartPoleLoggerDyna())
    agent.evaluate(max_iter=MAX_EPISODES)

    if agent.tensorboard:
        agent.tensorboard.flush()

In [ ]:
def main() -> None:
    """Sweep over plan steps and warmup periods with fixed hyperparameters."""
    env = ScaledCartPole(gym.make("CartPole-v1"))

    for plan_steps in MODEL_PLAN_STEPS:
        for warmup in WARMUPS:
            run_experiment(env, plan_steps, warmup)

In [ ]:
main()

## Analysis 

### Policy and Q-Surface after 50 Episodes
![Alt text](initial_policy.png)

After 50 episodes the agent learns the correct convex shape — states closer to upright are worth more, with the origin as the peak. However the relative difference in Q-values between actions is heavily skewed. It can't distinguish when to push left vs right.

### Policy and Q-Surface after 800 Episodes
![Alt text](policy.png)

After 800 episodes the action split converges close to optimal — push left when leaning right, push right when leaning left. Learning rate decay was crucial; without it the max(Q, a') step causes Q-values to spiral due to [maximization bias](https://en.wikipedia.org/wiki/Reinforcement_learning#Maximization_bias).

### No Alpha Decay — Maximization Bias
![Alt text](no_alpha_decay.png)

Without aggressive LR decay, Q-estimates spiral and one action dominates everywhere. The tabletop shape of the Q-surface shows the network treats all non-edge states as equally good — a well-trained network would have gradual curvature. The fix is either decaying the learning rate or tuning gamma to compress the future reward range enough for the agent to distinguish good from bad states.